# California Housing — Random Forest Regression

## (a) Data Loading & Exploration

In [ ]:
import pandas as pd, numpy as np
from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing(as_frame=True)
df = housing.frame
print("Shape:", df.shape)
df.head(10)

**Analysis:** Dataset loaded successfully with 20,640 rows and 9 columns. First 10 rows show plausible values.

In [ ]:
df.dtypes

In [ ]:
df.isnull().sum()

**Analysis:** No missing values detected; all features are numeric, so no imputation needed.

In [ ]:
target="MedHouseVal"
print("Mean:",df[target].mean()); print("Median:",df[target].median())
print("Min:",df[target].min()); print("Max:",df[target].max())
import matplotlib.pyplot as plt, seaborn as sns
plt.figure(figsize=(8,4)); sns.histplot(df[target],bins=50,kde=True)
plt.title("Distribution of MedHouseVal"); plt.show()

**Analysis:** Target is right-skewed with a cap at 5.0 ($500k); mean exceeds median.

## (b) Descriptive Statistics & Relationships

In [ ]:
df.describe()

**Analysis:** Features have varied scales; MedInc and population show wide ranges.

In [ ]:
plt.figure(figsize=(10,8)); corr=df.corr()
sns.heatmap(corr,annot=True,cmap="coolwarm",fmt=".2f"); plt.title("Correlation Heatmap"); plt.show()
print("Top 3:",corr[target].drop(target).abs().sort_values(ascending=False).head(3))

**Analysis:** MedInc, AveRooms, HouseAge are most correlated with MedHouseVal.

In [ ]:
plt.figure(figsize=(7,5)); sns.scatterplot(x="MedInc",y=target,data=df,alpha=0.3); plt.title("MedInc vs MedHouseVal"); plt.show()

**Analysis:** Strong positive linear relationship between income and house value.

In [ ]:
plt.figure(figsize=(8,6))
sc=plt.scatter(df["Longitude"],df["Latitude"],c=df[target],cmap="viridis",s=5,alpha=0.5)
plt.colorbar(sc,label="MedHouseVal"); plt.title("Geographical scatter"); plt.xlabel("Longitude"); plt.ylabel("Latitude"); plt.show()

**Analysis:** Higher values cluster along the coast (San Francisco/Los Angeles).

## (c) Train/Test Split & Random Forest

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
X=df.drop(columns=[target]); y=df[target]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
print("Train:",X_train.shape,"Test:",X_test.shape)
configs={"A":{"n_estimators":50,"max_depth":5},"B":{"n_estimators":100,"max_depth":10},"C":{"n_estimators":200,"max_depth":None}}
results={}
for n,p in configs.items():
    rf=RandomForestRegressor(**p,random_state=42); rf.fit(X_train,y_train)
    pred=rf.predict(X_test); r2=r2_score(y_test,pred); results[n]=r2
    print(f"Config {n} {p}: R2={r2:.4f}")
best=max(results,key=results.get); print("Best:",best)

**Analysis:** Config C (200 trees, unlimited depth) achieves highest R2; deeper trees capture more variance.

## (d) Best Model Evaluation

In [ ]:
from sklearn.metrics import mean_absolute_error,mean_squared_error
bp=configs[best]; brf=RandomForestRegressor(**bp,random_state=42); brf.fit(X_train,y_train)
pred=brf.predict(X_test)
mae=mean_absolute_error(y_test,pred); mse=mean_squared_error(y_test,pred); rmse=np.sqrt(mse); r2=r2_score(y_test,pred)
print(f"MAE:{mae:.4f} MSE:{mse:.4f} RMSE:{rmse:.4f} R2:{r2:.4f}")

In [ ]:
plt.figure(figsize=(7,7)); plt.scatter(y_test,pred,alpha=0.3)
mn,mx=y_test.min(),y_test.max(); plt.plot([mn,mx],[mn,mx],"r--",label="y=x")
plt.xlabel("Actual"); plt.ylabel("Predicted"); plt.title("Predicted vs Actual"); plt.legend(); plt.show()

**Analysis:** Points cluster near y=x; some under-prediction at high values due to cap.

In [ ]:
imp=brf.feature_importances_
fd=pd.DataFrame({"feature":X.columns,"importance":imp}).sort_values("importance")
plt.figure(figsize=(8,5)); plt.barh(fd["feature"],fd["importance"]); plt.title("Feature Importance"); plt.show()
print("Most important:",fd.iloc[-1]["feature"])

**Analysis:** MedInc is the most important feature for predicting house value.